## Imports and data loading

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from torch.utils.data import DataLoader, TensorDataset

from models.architectures import (
    CNNLSTMModel,
    LSTMAttentionModel,
    LSTMModel,
    TransformerModel,
    build_model,
)

repo_root = os.path.abspath('../')
with open(os.path.join(repo_root, 'config.yaml')) as f:
    cfg = yaml.safe_load(f)

splits_dir = os.path.join(repo_root, cfg['paths']['splits_dir'])
model_dir  = os.path.join(repo_root, cfg['paths']['model_dir'])

# ── Version config ────────────────────────────────────────────────────────────
# Switch DATASET to select which dataset to tune on.
#   'uhlrich'              -> Uhlrich_v{uhlrich_version}
#   'silder_mixed'         -> Silder_mixed_v{silder_version}
#   'silder_uhlrich_mixed' -> Silder_Uhlrich_mixed_v{silder_version}
DATASET = 'silder_mixed'

_uv = cfg['active']['uhlrich_version']
_sv = cfg['active']['silder_version']

if DATASET == 'uhlrich':
    _full, _major, _label = _uv, _uv.split('.')[0], 'Uhlrich'
elif DATASET == 'silder_mixed':
    _full, _major, _label = _sv, _sv.split('.')[0], 'Silder_mixed'
else:
    _full, _major, _label = _sv, _sv.split('.')[0], 'Silder_Uhlrich_mixed'

output_prefix   = f'{_label}_v{_full}'    # weights and Optuna study names
_data_prefix    = f'{_label}_v{_major}'   # splits (major version only)
_optuna_db_path = os.path.join(model_dir, f'optuna_v{_major}.db')
# ─────────────────────────────────────────────────────────────────────────────

train_data = np.load(os.path.join(splits_dir, f'{_data_prefix}_train_data.npz'))
val_data   = np.load(os.path.join(splits_dir, f'{_data_prefix}_val_data.npz'))
test_data  = np.load(os.path.join(splits_dir, f'{_data_prefix}_test_data.npz'),
                     allow_pickle=True)

X_train, y_train = train_data['X_train'], train_data['y_train']
X_val,   y_val   = val_data['X_val'],     val_data['y_val']
X_test,  y_test  = test_data['X_test'],   test_data['y_test']

INPUT_KEYS  = cfg['signals']['inputs']
OUTPUT_KEYS = [str(k) for k in test_data['output_keys']]

N_INPUTS  = len(INPUT_KEYS)
N_OUTPUTS = len(OUTPUT_KEYS)

MUSCLE_KEYS = [k for k in OUTPUT_KEYS if not k.startswith(('knee', 'ankle'))]
JOINT_KEYS  = [k for k in OUTPUT_KEYS if k.startswith(('knee', 'ankle'))]

print(f'Inputs  ({N_INPUTS}):  {INPUT_KEYS}')
print(f'Outputs ({N_OUTPUTS}): {OUTPUT_KEYS}')

assert X_train.shape[2] == N_INPUTS,  f'Expected {N_INPUTS} inputs,  got {X_train.shape[2]}'
assert y_train.shape[2] == N_OUTPUTS, f'Expected {N_OUTPUTS} outputs, got {y_train.shape[2]}'

SEQ_LEN = X_train.shape[1]
print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')

In [ ]:
for model_name in ['lstm', 'lstm_attn', 'cnn_lstm', 'transformer']:
    try:
        optuna.delete_study(
            study_name=f'{output_prefix}_{model_name}',
            storage=f'sqlite:///{_optuna_db_path}',
        )
        print(f'Deleted {model_name}')
    except KeyError:
        print(f'{model_name} not found, skipping')


## Quick data visualization

In [ ]:
perc_stance  = np.linspace(0, 1, SEQ_LEN)
achilles_idx = OUTPUT_KEYS.index('achilles')
fig, ax = plt.subplots(figsize=(10, 10))
for i in range(len(y_train)):
    ax.plot(perc_stance, y_train[i, :, achilles_idx], linewidth=0.8, alpha=0.5)
ax.set_title('Achilles — all training segments')
ax.set_xlabel('Percent Normalized Stance')
ax.set_ylabel('Force (N)')
plt.tight_layout()
plt.show()

## Device setup and tensor construction

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32).to(device)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).to(device)

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset   = TensorDataset(X_val_t,   y_val_t)

## Generic training loop

In [ ]:
LOSS = 'mse'   # 'mse' | 'peak_weighted'  — must match the active version in config.yaml

def peak_weighted_loss(y_pred, y_true):
    abs_true = y_true.abs()
    weights = abs_true / (abs_true.max(dim=1, keepdim=True).values + 1e-8)
    return ((y_pred - y_true) ** 2 * weights).mean()

criterion = nn.MSELoss()
_loss_fn  = peak_weighted_loss if LOSS == 'peak_weighted' else criterion


def train_eval(model, train_dataset, val_dataset,
               learning_rate, batch_size, regularization,
               grad_clip=0.0, num_epochs=500, patience=10):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate,
                           weight_decay=regularization)

    best_val_loss = float('inf')
    best_state    = None
    no_improve    = 0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = _loss_fn(model(X_batch), y_batch)
            loss.backward()
            if grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                val_loss += _loss_fn(model(X_batch), y_batch).item() * X_batch.size(0)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve    = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    model.load_state_dict(best_state)
    return best_val_loss, model

## Optuna objective functions (one per model)

In [ ]:
def objective_lstm(trial):
    hidden_size   = trial.suggest_categorical('hidden_size', [64, 128, 256, 512])
    num_layers    = trial.suggest_int('num_layers', 1, 4)
    dropout_rate  = trial.suggest_float('dropout_rate', 0.0, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    weight_decay  = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    grad_clip     = trial.suggest_categorical('grad_clip', [0.0, 0.5, 1.0, 2.0])

    model = LSTMModel(N_INPUTS, hidden_size, num_layers, N_OUTPUTS, dropout_rate).to(device)
    val_loss, _ = train_eval(model, train_dataset, val_dataset,
                             learning_rate, batch_size, weight_decay,
                             grad_clip=grad_clip, num_epochs=500, patience=10)
    return val_loss


def objective_lstm_attn(trial):
    hidden_size   = trial.suggest_categorical('hidden_size', [64, 128, 256, 512])
    num_layers    = trial.suggest_int('num_layers', 1, 4)
    # num_heads must divide hidden_size
    num_heads     = trial.suggest_categorical('num_heads', [2, 4, 8, 16])
    if hidden_size % num_heads != 0:
        raise optuna.TrialPruned()
    lstm_dropout  = trial.suggest_float('lstm_dropout', 0.0, 0.5)
    attn_dropout  = trial.suggest_float('attn_dropout', 0.0, 0.4)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    weight_decay  = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    grad_clip     = trial.suggest_categorical('grad_clip', [0.0, 0.5, 1.0, 2.0])

    model = LSTMAttentionModel(N_INPUTS, hidden_size, num_layers, num_heads, N_OUTPUTS,
                               lstm_dropout, attn_dropout).to(device)
    val_loss, _ = train_eval(model, train_dataset, val_dataset,
                             learning_rate, batch_size, weight_decay,
                             grad_clip=grad_clip, num_epochs=500, patience=10)
    return val_loss


def objective_cnn_lstm(trial):
    cnn_channels  = trial.suggest_categorical('cnn_channels', [32, 64, 128])
    hidden_size   = trial.suggest_categorical('hidden_size', [64, 128, 256, 512])
    num_layers    = trial.suggest_int('num_layers', 1, 4)
    dropout_rate  = trial.suggest_float('dropout_rate', 0.0, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    weight_decay  = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    grad_clip     = trial.suggest_categorical('grad_clip', [0.0, 0.5, 1.0, 2.0])

    model = CNNLSTMModel(N_INPUTS, cnn_channels, hidden_size, num_layers, N_OUTPUTS,
                         dropout_rate).to(device)
    val_loss, _ = train_eval(model, train_dataset, val_dataset,
                             learning_rate, batch_size, weight_decay,
                             grad_clip=grad_clip, num_epochs=500, patience=10)
    return val_loss


def objective_transformer(trial):
    d_model    = trial.suggest_categorical('d_model', [32, 48, 64, 96, 128, 192, 256])
    num_heads  = trial.suggest_categorical('num_heads', [2, 4, 8, 16])
    if d_model % num_heads != 0:
        raise optuna.TrialPruned()
    num_encoder_layers = trial.suggest_int('num_encoder_layers', 2, 12)
    ff_mult            = trial.suggest_categorical('ff_mult', [2, 3, 4, 6, 8])
    dim_feedforward    = ff_mult * d_model
    dropout_rate       = trial.suggest_float('dropout_rate', 0.0, 0.4)
    learning_rate      = trial.suggest_float('learning_rate', 5e-5, 5e-3, log=True)
    weight_decay       = trial.suggest_float('weight_decay', 1e-6, 5e-2, log=True)
    batch_size         = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    grad_clip          = trial.suggest_categorical('grad_clip', [0.0, 0.5, 1.0, 2.0])

    model = TransformerModel(N_INPUTS, N_OUTPUTS, d_model, num_heads,
                             num_encoder_layers, dim_feedforward, dropout_rate).to(device)
    val_loss, _ = train_eval(model, train_dataset, val_dataset,
                             learning_rate, batch_size, weight_decay,
                             grad_clip=grad_clip, num_epochs=500, patience=10)
    return val_loss


print('Objective functions defined.')

## Model queue
Edit `MODELS_TO_TUNE` and `N_TRIALS` below, then run this cell to tune all selected models back-to-back.

In [ ]:
import traceback
 
# ── Configure the queue ──────────────────────────────────────────────────────
MODELS_TO_TUNE = ['lstm', 'lstm_attn', 'cnn_lstm', 'transformer']  # remove any you don't want
N_TRIALS       = 100   # Optuna trials per model
FINAL_EPOCHS   = 1000  # epochs for the final retrain with best params
FINAL_PATIENCE = 20
# ─────────────────────────────────────────────────────────────────────────────

OBJECTIVE_MAP = {
    'lstm':        objective_lstm,
    'lstm_attn':   objective_lstm_attn,
    'cnn_lstm':    objective_cnn_lstm,
    'transformer': objective_transformer,
}


results = {}   # stores {model_name: {'study': ..., 'model': ..., 'val_loss': ...}}

for model_name in MODELS_TO_TUNE:
    print(f'\n{"="*60}')
    print(f'  Tuning: {model_name}  ({N_TRIALS} trials)')
    print(f'{"="*60}')

    try:
        study = optuna.create_study(
            study_name=f'{output_prefix}_{model_name}',
            direction='minimize',
            storage=f'sqlite:///{_optuna_db_path}',
            load_if_exists=True,   # resumes if crashed
        )
        study.optimize(OBJECTIVE_MAP[model_name], n_trials=N_TRIALS)

        bp = study.best_params
        print(f'\nBest params: {bp}')
        print(f'Best val loss: {study.best_value:.4f}')

        # Final retrain with more epochs
        print('Retraining final model...')
        final_model = build_model(model_name, bp, N_INPUTS, N_OUTPUTS, device)
        final_val_loss, final_model = train_eval(
            final_model, train_dataset, val_dataset,
            learning_rate=bp.get('learning_rate'),
            batch_size=bp.get('batch_size', 32),
            regularization=bp.get('weight_decay'),
            grad_clip=bp.get('grad_clip', 0.0),
            num_epochs=FINAL_EPOCHS,
            patience=FINAL_PATIENCE,
        )
        save_path = os.path.join(model_dir, f'{output_prefix}_{model_name}.pth')
        torch.save(final_model.state_dict(), save_path)
        print(f'Saved → {save_path}  (final val loss: {final_val_loss:.4f})')

        results[model_name] = {'study': study, 'model': final_model,
                               'val_loss': final_val_loss}

        # Free GPU memory before next model
        del final_model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f'ERROR tuning {model_name}: {e}')
        traceback.print_exc()

print('\n=== Queue complete ===')
for name, r in results.items():
    print(f'  {name}: val_loss = {r["val_loss"]:.4f}')

## Test evaluation — reload saved models and score on test set

In [ ]:
criterion = nn.MSELoss()
test_results = {}

for model_name in MODELS_TO_TUNE:
    save_path = os.path.join(model_dir, f'{output_prefix}_{model_name}.pth')
    try:
        study   = results[model_name]['study']
        bp      = study.best_params
        model   = build_model(model_name, bp, N_INPUTS, N_OUTPUTS, device)
        model.load_state_dict(torch.load(save_path, map_location=device))
        model.eval()

        with torch.no_grad():
            preds     = model(X_test_t)
            test_loss = criterion(preds, y_test_t).item()

        test_results[model_name] = {'model': model, 'preds': preds, 'test_loss': test_loss}
        print(f'{model_name}: test_loss = {test_loss:.4f}')

    except Exception as e:
        print(f'Could not evaluate {model_name}: {e}')

## Visualize predictions for each model — muscle forces

In [ ]:
SAMPLE_IDX    = 0
muscle_idxs   = [OUTPUT_KEYS.index(k) for k in MUSCLE_KEYS]
joint_idxs    = [OUTPUT_KEYS.index(k) for k in JOINT_KEYS]
true_np       = y_test_t[SAMPLE_IDX].cpu().numpy()

for model_name, r in test_results.items():
    pred_np = r['preds'][SAMPLE_IDX].cpu().numpy()

    # ── muscles ──
    ncols = 4
    nrows = math.ceil(len(muscle_idxs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = axes.flatten()
    for j, idx in enumerate(muscle_idxs):
        axes[j].plot(perc_stance, true_np[:, idx], label='True')
        axes[j].plot(perc_stance, pred_np[:, idx], label='Pred', linestyle='dashed')
        axes[j].set_title(OUTPUT_KEYS[idx])
        axes[j].set_xlabel('% Stance')
        axes[j].set_ylabel('Force (N)')
        axes[j].legend(fontsize=7)
    for j in range(len(muscle_idxs), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle(f'{model_name} — Muscle Forces (test[{SAMPLE_IDX}])  loss={r["test_loss"]:.4f}', fontsize=13)
    plt.tight_layout()
    plt.show()

    # ── joint contact forces ──
    ncols = 3
    nrows = math.ceil(len(joint_idxs) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = axes.flatten()
    for j, idx in enumerate(joint_idxs):
        axes[j].plot(perc_stance, true_np[:, idx], label='True')
        axes[j].plot(perc_stance, pred_np[:, idx], label='Pred', linestyle='dashed')
        axes[j].set_title(OUTPUT_KEYS[idx])
        axes[j].set_xlabel('% Stance')
        axes[j].set_ylabel('Force (N)')
        axes[j].legend(fontsize=7)
    for j in range(len(joint_idxs), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle(f'{model_name} — Joint Contact Forces (test[{SAMPLE_IDX}])  loss={r["test_loss"]:.4f}', fontsize=13)
    plt.tight_layout()
    plt.show()